# 10. Прогнозная модель: LightGBM
Тот же датасет, то же разбиение и те же метрики, что в ноутбуке 9 — сравниваем бустинг с базовыми решениями.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import RocCurveDisplay
from src.build_dataset import get_dataset, FEATURES, TARGET
from src.modeling import split_by_competition, evaluate, baseline_models

df = get_dataset()
train, test = split_by_competition(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

Пропуски LightGBM обрабатывает сам. Параметры пока стандартные, подбор — следующий шаг.

In [ ]:
lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                      is_unbalance=True, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)

## Сравнение с базовыми решениями

In [ ]:
models = {**baseline_models(), "lightgbm": lgbm}
results = {"rule_experience": evaluate(y_test, X_test["max_prev_comps"].fillna(0))}
for name, model in models.items():
    if name != "lightgbm":
        model.fit(X_train, y_train)
    results[name] = evaluate(y_test, model.predict_proba(X_test)[:, 1])

pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name in ["logreg", "lightgbm"]:
    RocCurveDisplay.from_estimator(models[name], X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
plt.savefig("../reports/figures/10_roc_curves.png", bbox_inches="tight")
plt.show()

## Важность признаков

In [ ]:
importance = pd.Series(lgbm.booster_.feature_importance("gain"), index=FEATURES).sort_values()
importance.plot(kind="barh", figsize=(7, 4), title="Важность признаков (gain)")
plt.savefig("../reports/figures/10_feature_importance.png", bbox_inches="tight")
plt.show()

TODO: подбор параметров (кросс-валидация по соревнованиям), признаки за 2-ю и 3-ю недели.
Помнить: max_tier — текущий уровень участника, частично «знает» будущее.